In [13]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [15]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,legacyLink,tags.data,associatedGroups.data,source,hostName,dnsActive,whoisActive,address,url,indicator
0,5629499542008918,2025-05-19T11:57:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:35:49Z,3.0,69,RITM0581780/TASK0877793,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,137.184.174.181
1,6755399459078380,2025-06-25T12:17:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:35:48Z,3.0,74,TASK0891299,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,143.110.213.190
2,6755399465229481,2025-07-28T17:31:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:35:47Z,3.0,79,TASK0902923,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,176.65.148.80
3,6755399465229477,2025-07-28T17:31:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:33:38Z,3.0,79,TASK0902923,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.94.95.132
4,6755399460007552,2025-07-02T12:05:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T11:01:15Z,4.0,69,"Details\nIn late June 2025, Iran-aligned hackt...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,197.219.228.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1316,4503719,2024-01-23T19:14:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:43Z,3.0,80,A phishing email with a .htm file containing a...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 238537, 'name': 'fake voicemail', 'las...","[{'id': 308984, 'dateAdded': '2024-01-23T19:14...",https://aka.ms/LearnAboutSenderIdentification,NaN,NaN,NaN,NaN,aka.ms/learnaboutsenderidentification,aka.ms/learnaboutsenderidentification
1317,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp
1319,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,https://google,NaN,NaN,NaN,NaN,google,google
1320,4517160,2024-02-05T17:10:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-05T23:23:34Z,3.0,67,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35057, 'name': 'FDA Splunk API', 'last...","[{'id': 313616, 'dateAdded': '2024-02-05T16:18...",https://realinvestmentadvice.com/,NaN,NaN,NaN,NaN,realinvestmentadvice.com/,realinvestmentadvice.com/


In [17]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

display(observed_src_with_tags[['indicator', 'tag_list']])

,indicator,tag_list
0,137.184.174.181,"[DHA Splunk API, OS Splunk API, VA CSOC CTS Sp..."
1,143.110.213.190,"[DHA Splunk API, OS Splunk API, CMS Splunk API..."
2,176.65.148.80,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
3,80.94.95.132,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
4,197.219.228.62,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S..."
...,...,...
1312,aka.ms/learnaboutsenderidentification,"[fake voicemail, .htm, Alert ID : 1e4e8308, Cr..."
1313,geo.netsupportsoftware.com/location/loca.asp,"[business email compromise, hospital, Observed..."
1314,google,"[OS Splunk API, HRSA Splunk API, Observed]"
1315,realinvestmentadvice.com/,"[FDA Splunk API, Observed, CDC Splunk API]"


In [19]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,2
1,DDoS,136
2,Spam,0
3,Phishing,19
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,14
7,Data Theft,9
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [25]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,tags.data,associatedGroups.data,source,hostName,dnsActive,whoisActive,address,url,indicator,Botnet
0,5629499542008918,2025-05-19T11:57:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:35:49Z,3.0,69,RITM0581780/TASK0877793,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,137.184.174.181,[]
1,6755399459078380,2025-06-25T12:17:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:35:48Z,3.0,74,TASK0891299,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,143.110.213.190,[]
2,6755399465229481,2025-07-28T17:31:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:35:47Z,3.0,79,TASK0902923,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,176.65.148.80,[]
3,6755399465229477,2025-07-28T17:31:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T12:33:38Z,3.0,79,TASK0902923,...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.94.95.132,[]
4,6755399460007552,2025-07-02T12:05:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T11:01:15Z,4.0,69,"Details\nIn late June 2025, Iran-aligned hackt...",...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,197.219.228.62,[DDoS]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1316,4503719,2024-01-23T19:14:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:43Z,3.0,80,A phishing email with a .htm file containing a...,...,"[{'id': 238537, 'name': 'fake voicemail', 'las...","[{'id': 308984, 'dateAdded': '2024-01-23T19:14...",https://aka.ms/LearnAboutSenderIdentification,NaN,NaN,NaN,NaN,aka.ms/learnaboutsenderidentification,aka.ms/learnaboutsenderidentification,[]
1317,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,"[{'id': 34822, 'name': 'business email comprom...","[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,geo.netsupportsoftware.com/location/loca.asp,[]
1319,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,https://google,NaN,NaN,NaN,NaN,google,google,[]
1320,4517160,2024-02-05T17:10:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-05T23:23:34Z,3.0,67,NaN,...,"[{'id': 35057, 'name': 'FDA Splunk API', 'last...","[{'id': 313616, 'dateAdded': '2024-02-05T16:18...",https://realinvestmentadvice.com/,NaN,NaN,NaN,NaN,realinvestmentadvice.com/,realinvestmentadvice.com/,[]


In [22]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_24412\1319479530.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250807.csv']"

'Loaded data from 1 files.'

In [29]:
import pandas as pd

df = observed_src.copy()

df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

groups_exploded = (
    df[['indicator', 'associatedGroups.data']]
      .explode('associatedGroups.data')
      .dropna(subset=['associatedGroups.data'])
)

group_norm = pd.json_normalize(
    groups_exploded['associatedGroups.data']
).rename(columns={'id':'group_id','name':'group_name'})

groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

group_agg = (
    groups_exploded
      .drop_duplicates(subset=['indicator','group_id','group_name'])
      .groupby('indicator', as_index=False)
      .agg({
          'group_id':   lambda ids: list(ids),
          'group_name': lambda names: list(names)
      })
      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','hostName','dnsActive','whoisActive','source',
    'address','url'
]

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c not in ['address','ip','source','url','legacyLink']]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
for col in ['partners','group_ids','group_names'] + tag_fields:
    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,tag_lastUsed,tag_description,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners,group_ids,group_names
0,1.4.195.14,6755399460007413,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T09:35:19Z,4.0,69,...,"2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 20...",,,,,,,,5629499544001057,INDICATOR NOTICE 25178.1 – Iran-Aligned Hackti...
1,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-05T07:41:51Z,3.0,70,...,"2025-08-07T12:35:49Z, 2025-08-07T12:25:56Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",,
2,103.133.60.12,6755399460008266,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-02T07:38:32Z,4.0,69,...,"2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 20...",,,,,,,"DHA, FDA",5629499544001057,INDICATOR NOTICE 25178.1 – Iran-Aligned Hackti...
3,103.149.86.208,6755399458556969,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-06T07:45:09Z,3.0,72,...,"2025-08-07T12:35:49Z, 2025-08-07T12:25:56Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS",,
4,103.171.255.188,6755399460007704,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T06:42:35Z,4.0,69,...,"2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 20...",,,,,,,"DHA, FDA",5629499544001057,INDICATOR NOTICE 25178.1 – Iran-Aligned Hackti...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
835,vtext.com,5182028,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-05T07:42:02Z,3.0,46,...,"2025-08-07T12:35:49Z, 2025-08-06T18:49:57Z, 20...",Adversaries may send phishing messages to gain...,T1566,['Initial Access'],1.0,"['Linux', 'macOS', 'Windows', 'SaaS', 'Identit...",6.0,"DHA, NIH, IHS",535913,NIH Phishing Emails 11/27/2024
836,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-05T07:42:11Z,3.0,83,...,"2025-08-07T12:35:49Z, 2025-01-31T14:04:11Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS","548890, 548570","HTOC-20250131-0903-A, Blocked URLs by NIH for ..."
837,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-06T07:45:30Z,3.0,81,...,"2025-08-07T12:35:49Z, 2025-01-31T14:04:11Z, 20...",,,,,,,"DHA, CMS, NIH, IHS",548570,Blocked URLs by NIH for DeepSeek
838,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-06T07:45:29Z,4.0,49,...,"2025-08-07T12:35:49Z, 2025-08-07T12:25:56Z, 20...",Observations reported by the HHS Ofice of the ...,,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS","332131, 331131","HTOC-20240423-0923-A, VA Hosts Reaching Out to..."


In [30]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

display(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        display(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        display("Enrichment column not found; nothing to flatten.")

# Optional: report/log failures
if failed:
    display(f"{len(failed)} indicators failed to enrich.")
    # Example: pd.DataFrame(failed).to_json("enrich_failures.json", orient="records")


'Enriching 840 indicators with Shodan & VirusTotalV3...'

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host arpdcresources.ca cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host bimbinlombardia.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress cameron@yourpensionmeeting.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL careersinpsychology.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"More than one indicator matches the criteria you specified. Try looking up by ID instead.","status":"Error"}

'Successfully enriched and merged 812 indicators.'

'28 indicators failed to enrich.'

In [31]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1.4.195.14,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T09:35:19Z,4.0,69,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Khueang Nai],[Thailand],"[nt-isp.net, totinternet.net]","[node-d8u.pool-1-4.dynamic.totinternet.net, no...",[TOT Public Company Limited],"[{'transport': 'udp', 'port': 161, 'product': ...",[TOT Public Company Limited],None,None,1.0
1,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-05T07:41:51Z,3.0,70,INC9067814,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,6.0
2,103.133.60.12,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-02T07:38:32Z,4.0,69,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Terbanggi Besar],[Indonesia],None,None,[PT Tunas Link Indonesia],"[{'transport': 'tcp', 'port': 53, 'data': ' Re...",[PT Tunas Link Indonesia],None,None,1.0
3,103.149.86.208,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-06T07:45:09Z,3.0,72,INC9097535,...,[Hanoi],[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],None,5.0
4,103.171.255.188,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T06:42:35Z,4.0,69,"Details\nIn late June 2025, Iran-aligned hackt...",...,[Surakarta],[Indonesia],None,None,[PT Zona Kolektif Indonesia],"[{'transport': 'tcp', 'port': 2000, 'product':...",[PT Zona Kolektif Indonesia],None,None,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
835,vtext.com,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-05T07:42:02Z,3.0,46,The correlation search is based on an automate...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
836,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-05T07:42:11Z,3.0,83,Executive Summary: \n\nThe following DeepSeek...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
837,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-06T07:45:30Z,3.0,81,The following list of urls and domains are cur...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
838,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-06T07:45:29Z,4.0,49,The Department of Veterans Affairs (VA) receiv...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Scoring_Data"]
collection = db["enriched_observation_Data"]

# Convert DataFrame to dictionary records
records = recent_tags.to_dict(orient="records")

# Insert records into MongoDB
result = collection.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} documents into enriched_observation_Data.")

client.close()


Inserted 840 documents into enriched_observation_Data.


In [84]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Scoring_Data"]
collection = db["enriched_observation_Data"]

# Pull data back from MongoDB
docs = list(collection.find())
recent_tags = pd.DataFrame(docs)
print(f"Pulled {len(recent_tags)} documents from enriched_observation_Data.")


Pulled 840 documents from enriched_observation_Data.


In [85]:
recent_tags

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,6894b04f869107366612ccaf,1.4.195.14,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T09:35:19Z,4.0,69,...,[Khueang Nai],[Thailand],"[nt-isp.net, totinternet.net]","[node-d8u.pool-1-4.dynamic.totinternet.net, no...",[TOT Public Company Limited],"[{'transport': 'udp', 'port': 161, 'product': ...",[TOT Public Company Limited],None,None,1.0
1,6894b04f869107366612ccb0,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-05T07:41:51Z,3.0,70,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,6.0
2,6894b04f869107366612ccb1,103.133.60.12,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-02T07:38:32Z,4.0,69,...,[Terbanggi Besar],[Indonesia],None,None,[PT Tunas Link Indonesia],"[{'transport': 'tcp', 'port': 53, 'data': ' Re...",[PT Tunas Link Indonesia],None,None,1.0
3,6894b04f869107366612ccb2,103.149.86.208,2025-06-11T14:36:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-06T07:45:09Z,3.0,72,...,[Hanoi],[Viet Nam],None,None,[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],"[{'transport': 'tcp', 'port': 22, 'product': '...",[4S TECHNOLOGY TRADING SERVICES COMPANY LIMITED],[eol-product],None,5.0
4,6894b04f869107366612ccb3,103.171.255.188,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-07T06:42:35Z,4.0,69,...,[Surakarta],[Indonesia],None,None,[PT Zona Kolektif Indonesia],"[{'transport': 'tcp', 'port': 2000, 'product':...",[PT Zona Kolektif Indonesia],None,None,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
835,6894b04f869107366612cff2,vtext.com,2024-12-16T18:58:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-05T07:42:02Z,3.0,46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
836,6894b04f869107366612cff3,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-05T07:42:11Z,3.0,83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
837,6894b04f869107366612cff4,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-06T07:45:30Z,3.0,81,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
838,6894b04f869107366612cff5,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-06T07:45:29Z,4.0,49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [108]:
import re

# helper to flatten list-or-str into a list of strings
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# regex: 
# ^[A-Za-z]+       → first label is one “word” (letters only)
# (?:\.[^.]+){3}$  → exactly three more “.” followed by non-dot characters
four_label_word_prefix = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')

def host_match(hosts):

    for h in flatten_hosts(hosts):
        if four_label_word_prefix.match(h):
            return True
    return False

matched = recent_tags[
    recent_tags['enrich_hostNames'].apply(host_match)
]

matched_results = matched[['indicator', 'enrich_hostNames']]
display(matched_results)


,indicator,enrich_hostNames
154,185.230.63.171,[unalocated.63.wixsite.com]
163,190.121.128.201,"[190121128201.mc.net.co, mail.mc.net.co]"
181,193.163.125.103,[felicitous.census.internet-measurement.com]
182,193.163.125.107,[hearty.census.internet-measurement.com]
183,193.163.125.108,[erudite.census.internet-measurement.com]
...,...,...
802,91.196.152.56,[otto.probe.onyphe.net]
803,91.196.152.66,[lancaster.probe.onyphe.net]
804,91.196.152.67,[stephen.probe.onyphe.net]
805,91.196.152.71,[cherise.probe.onyphe.net]


In [109]:
matched_results[matched_results['indicator'] == 'irp.cdn-website.com']

,indicator,enrich_hostNames


In [87]:
recent_tags['Botnet'].value_counts()

Botnet
[]                    770
[DDoS]                 62
[Phishing]              4
[ransomware]            3
[ransomware, DDoS]      1
Name: count, dtype: int64

In [111]:
recent_tags[recent_tags['indicator'] == 'irp.cdn-website.com']

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
824,6894b04f869107366612cfe7,irp.cdn-website.com,2025-04-20T17:12:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-08-07T07:44:03Z,3.0,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [114]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
import warnings
import re
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG: all adjustable weights & parameters live here
# ─────────────────────────────────────────────────────────────────────────────
# Temporal/volume
LAMBDA_DECAY = 0.10              # exponential decay for observations
LOG_CLIP_Q = 0.95                # upper clip quantile for log_decayed_obs
TEMP_Q_LOW = 0.25                # quantile for "light" recent sightings
TEMP_Q_HIGH = 0.75               # quantile for "heavy" recent sightings
BURST_Q_HIGH = 0.75              # quantile for "bursty" pattern

# Sources/trust
SRC_Q_LOW = 0.25                 # few corroborating sources
SRC_Q_HIGH = 0.75                # broad corroboration
SRC_WEIGHT_Q_HIGH = 0.75         # high total trust weight
HIGH_TRUST_THRESHOLD = 7         # owner trust cutoff for "high"

# False positives
FP_Q_LOW = 0.25
FP_Q_HIGH = 0.75
FP_RATE_LOW_FLAG = 0.10
FP_RATE_HIGH_FLAG = 0.50

# Context
PORT_Q_HIGH = 0.75
VT_Q_HIGH = 0.75                 # (used only in messaging)
VT_WEIGHT = 0.50                 # contribution of VT to combined score (0..1)

# Hostname soft penalty
HOST_PENALTY_WEIGHT = 0.15       # fractional downweight applied if pattern matches (e.g., 0.15 = -15%)

# Analyst signals
RATING_HIGH = 4                  # rating considered "high"
CONFIDENCE_HIGH = 80             # confidence considered "high"

# Continuity mapping
CONTINUITY_MAP = {
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}

# Modeling
FEATURE_COLS = [
    'continuity_score', 'log_decayed_obs', 'obs_trend_slope', 'obs_burstiness',
    'num_distinct_sources', 'sum_source_weight', 'fp_rate',
    'port_diversity', 'vt_malicious_count', 'port_scan_confidence',
    'botnet_flag', 'Xcrit', 'host_generic_4label_wordprefix'
]
MODEL_CONTAMINATION = 0.05
MODEL_RANDOM_STATE = 42

# Severity bins (do not change unless ranges are updated)
BINS_1000 = [0, 199, 500, 800, 1000]
BIN_LABELS = ['Low', 'Medium', 'High', 'Critical']

# Explanation composition
EXPLAIN_COLS = [
    'msg_temporal','msg_sources','msg_fp','msg_context',
    'msg_behavior','msg_continuity','msg_botnet','msg_xcrit',
    'msg_hostnames','msg_model','msg_final'
]

# Hostname pattern regex (letters-only first label + exactly 3 more labels → total 4)
FOUR_LABEL_WORD_PREFIX = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# ─────────────────────────────────────────────────────────────────────────────
# 1) Temporal & Volume Features
# ─────────────────────────────────────────────────────────────────────────────
def compute_temporal_volume_features(df, daily_counts_df, lambda_decay=None, today=None):
    if today is None:
        today = pd.Timestamp(datetime.utcnow().date())
    if lambda_decay is None:
        lambda_decay = LAMBDA_DECAY

    df = df.copy()
    df['dateAdded'] = pd.to_datetime(df['dateAdded'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
    df['lastObserved'] = pd.to_datetime(df['lastObserved'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()

    df['observations'] = pd.to_numeric(df.get('observations', 0), errors='coerce').fillna(0).clip(lower=0)
    df['raw_obs'] = df['observations']

    days = (today - df['dateAdded']).dt.days
    df['days_since_first_seen'] = days.where(days.notna(), 0).clip(lower=0).astype(int)

    df['decayed_obs'] = df['raw_obs'] * np.exp(-lambda_decay * df['days_since_first_seen'])
    df['log_decayed_obs'] = np.log1p(df['decayed_obs'])
    clip_val = df['log_decayed_obs'].quantile(LOG_CLIP_Q)
    df['log_decayed_obs'] = df['log_decayed_obs'].clip(upper=clip_val)

    # daily counts typing
    if not daily_counts_df.empty:
        dcd = daily_counts_df.copy()
        dcd['date'] = pd.to_datetime(dcd['date'], errors='coerce')
        dcd['count'] = pd.to_numeric(dcd['count'], errors='coerce').fillna(0)
    else:
        dcd = pd.DataFrame(columns=['indicator','date','count'])

    def trend_burst(ind):
        counts = dcd.loc[dcd['indicator'] == ind].sort_values('date')['count'].values
        if counts.size >= 2:
            x = np.arange(counts.size)
            slope = float(np.polyfit(x, counts, 1)[0])
            mean = counts.mean()
            burst = float(np.std(counts) / mean) if mean > 0 else 0.0
        else:
            slope, burst = 0.0, 0.0
        return slope, burst

    tb = df['indicator'].apply(trend_burst)
    df[['obs_trend_slope', 'obs_burstiness']] = pd.DataFrame(tb.tolist(), index=df.index)

    # messages
    q_obs = df['log_decayed_obs'].quantile([TEMP_Q_LOW, TEMP_Q_HIGH]).to_dict()
    q_burst = df['obs_burstiness'].quantile([BURST_Q_HIGH]).to_dict()
    def msg_temporal(r):
        msgs = []
        if r['log_decayed_obs'] >= q_obs[TEMP_Q_HIGH]:
            msgs.append("heavy recent sightings (high decayed volume)")
        elif r['log_decayed_obs'] <= q_obs[TEMP_Q_LOW]:
            msgs.append("light recent sightings (low decayed volume)")
        if r['obs_trend_slope'] > 0:
            msgs.append("upward trend in daily sightings")
        elif r['obs_trend_slope'] < 0:
            msgs.append("downward trend in daily sightings")
        if r['obs_burstiness'] >= q_burst[BURST_Q_HIGH]:
            msgs.append("bursty observation pattern")
        return "; ".join(msgs) or "stable/average observation pattern"
    df['msg_temporal'] = df.apply(msg_temporal, axis=1)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 2) Source Diversity & Trust
# ─────────────────────────────────────────────────────────────────────────────
def compute_source_diversity(df, source_sightings_df, trust_map):
    df = df.copy()
    ssd = source_sightings_df.copy()
    if ssd.empty:
        df['num_distinct_sources'] = 0
        df['sum_source_weight'] = 0
        df['max_source_weight'] = 0
        df['high_low_trust_ratio'] = 0.0
        df['msg_sources'] = "no corroborating sources"
        return df

    grp = ssd.groupby('indicator', sort=False)
    df['num_distinct_sources'] = grp['ownerId'].nunique().reindex(df['indicator']).fillna(0).astype(int).values

    owner_lists = grp['ownerId'].apply(list).to_dict()
    sums, maxs, ratios = [], [], []
    for ind in df['indicator']:
        owners = owner_lists.get(ind, [])
        weights = [trust_map.get(o, 5) for o in owners]
        sums.append(sum(weights))
        maxs.append(max(weights) if weights else 0)
        high = sum(w >= HIGH_TRUST_THRESHOLD for w in weights)
        low = sum(w < HIGH_TRUST_THRESHOLD for w in weights)
        ratios.append(high / (low if low > 0 else 1))
    df['sum_source_weight'] = sums
    df['max_source_weight'] = maxs
    df['high_low_trust_ratio'] = ratios

    q_src = df['num_distinct_sources'].quantile([SRC_Q_LOW, SRC_Q_HIGH]).to_dict()
    q_weight = df['sum_source_weight'].quantile([SRC_WEIGHT_Q_HIGH]).to_dict()
    def msg_sources(r):
        msgs = []
        if r['num_distinct_sources'] >= q_src[SRC_Q_HIGH]:
            msgs.append("broad corroboration across sources")
        elif r['num_distinct_sources'] <= q_src[SRC_Q_LOW]:
            msgs.append("limited corroboration")
        if r['sum_source_weight'] >= q_weight[SRC_WEIGHT_Q_HIGH]:
            msgs.append("supported by high‑trust owners")
        if r['high_low_trust_ratio'] >= 1.0 and r['num_distinct_sources'] > 0:
            msgs.append("high‑trust > low‑trust ratio")
        return "; ".join(msgs) or "moderate corroboration"
    df['msg_sources'] = df.apply(msg_sources, axis=1)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 3) False-Positive Signals
# ─────────────────────────────────────────────────────────────────────────────
def compute_fp_signals(df):
    df = df.copy()
    if 'falsePositiveCount' not in df.columns:
        df['falsePositiveCount'] = 0
    df['falsePositiveCount'] = pd.to_numeric(df['falsePositiveCount'], errors='coerce').fillna(0).clip(lower=0)
    obs = df.get('observations', pd.Series(0, index=df.index)).replace(0, 1)
    df['fp_count'] = df['falsePositiveCount']
    df['fp_rate'] = (df['fp_count'] / obs).clip(lower=0)

    q_fp = df['fp_rate'].quantile([FP_Q_LOW, FP_Q_HIGH]).to_dict()
    def msg_fp(r):
        if r['fp_rate'] >= q_fp[FP_Q_HIGH] and r['fp_rate'] > 0:
            return "elevated false‑positive rate (down‑weights score)"
        if r['fp_rate'] <= q_fp[FP_Q_LOW]:
            return "low false‑positive rate"
        return "moderate false‑positive rate"
    df['msg_fp'] = df.apply(msg_fp, axis=1)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 4) Contextual Metadata (ports, VT, hostnames)
# ─────────────────────────────────────────────────────────────────────────────
def compute_context_metadata(df):
    df = df.copy()
    types = df.get('type')
    if types is not None:
        df = pd.concat([df, pd.get_dummies(types, prefix='type', dtype=int)], axis=1)

    ports = df.get('enrich_openPorts')
    if ports is None:
        df['port_diversity'] = 0
    else:
        def port_count(p):
            if isinstance(p, (list, set, tuple)):
                return len(set([str(x) for x in p]))
            if isinstance(p, str):
                return len(set([x.strip() for x in p.split(',') if x.strip()]))
            return 0
        df['port_diversity'] = ports.apply(port_count).astype(int)

    df['vt_malicious_count'] = pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce').fillna(0).clip(lower=0).astype(int)

    q_port = df['port_diversity'].quantile([PORT_Q_HIGH]).to_dict()
    q_vt = df['vt_malicious_count'].quantile([VT_Q_HIGH]).to_dict()
    def msg_context(r):
        msgs = []
        if r['port_diversity'] >= q_port.get(PORT_Q_HIGH, 0) and r['port_diversity'] > 0:
            msgs.append("broad port activity observed")
        if r['vt_malicious_count'] >= q_vt.get(VT_Q_HIGH, 0) and r['vt_malicious_count'] > 0:
            msgs.append("strong VT malicious consensus")
        elif r['vt_malicious_count'] == 0:
            msgs.append("no VT malicious detections")
        return "; ".join(msgs) or "limited contextual signals"
    df['msg_context'] = df.apply(msg_context, axis=1)

    # Hostname pattern
    def host_match(hosts):
        for h in flatten_hosts(hosts):
            if FOUR_LABEL_WORD_PREFIX.match(h):
                return True
        return False

    host_col = df.get('enrich_hostNames', None)
    if host_col is None:
        df['host_generic_4label_wordprefix'] = 0
        df['msg_hostnames'] = "no hostnames"
    else:
        df['host_generic_4label_wordprefix'] = host_col.apply(host_match).astype(int)
        df['msg_hostnames'] = np.where(
            df['host_generic_4label_wordprefix'] == 1,
            "hostname pattern suggests generic 4‑label prefix (down‑weighted)",
            "no generic 4‑label hostname pattern"
        )
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 5) Behavioral Signals
# ─────────────────────────────────────────────────────────────────────────────
def compute_behavioral(df):
    df = df.copy()
    tags = df.get('enrich_tags')
    if tags is None:
        df['port_scan_confidence'] = 0
    else:
        def has_scanner(x):
            if isinstance(x, (list, set, tuple)):
                s = ' '.join(map(str, x))
            else:
                s = str(x)
            return int('scanner' in s.lower())
        df['port_scan_confidence'] = tags.apply(has_scanner).astype(int)

    df['msg_behavior'] = np.where(df['port_scan_confidence'] > 0,
                                  "scanner behavior present",
                                  "no scanner behavior")
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 6) Continuity Score
# ─────────────────────────────────────────────────────────────────────────────
def compute_continuity_score(df):
    df = df.copy()
    df['continuity_score'] = df.get('type', '').map(CONTINUITY_MAP).fillna(0).astype(int)
    def msg_cont(r):
        if r['continuity_score'] == 3:
            return "hash‑level continuity (long‑lived indicator)"
        if r['continuity_score'] == 2:
            return "name/url continuity (medium‑lived indicator)"
        if r['continuity_score'] == 1:
            return "IP continuity (short‑lived indicator)"
        return "unknown continuity"
    df['msg_continuity'] = df.apply(msg_cont, axis=1)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 7) Botnet Flag
# ─────────────────────────────────────────────────────────────────────────────
def compute_botnet_flag(df):
    botnet_actions = {'Scanning','DDoS','Spam','Phishing','Cryptojacking','Credential Stuffing'}
    df = df.copy()
    col = df.get('Botnet', None)

    if col is None:
        df['botnet_flag'] = 0
    else:
        def is_botnet(val):
            if isinstance(val, (list, set, tuple)):
                return int(bool(set(map(str, val)) & botnet_actions))
            if isinstance(val, str):
                parts = {p.strip() for p in val.replace(',', ' ').split()}
                return int(bool(parts & botnet_actions))
            return 0
        df['botnet_flag'] = col.apply(is_botnet).astype(int)

    df['msg_botnet'] = np.where(df['botnet_flag'] > 0,
                                "botnet‑style activity reported",
                                "no botnet activity")
    return df

# ─────────────────────────────────────────────────────────────────────────────
# Feature Assembly
# ─────────────────────────────────────────────────────────────────────────────
def compute_features(df_raw, daily_counts_df, source_sightings_df, trust_map):
    df = compute_temporal_volume_features(df_raw, daily_counts_df)
    df = compute_source_diversity(df, source_sightings_df, trust_map)
    df = compute_fp_signals(df)
    df = compute_context_metadata(df)
    df = compute_behavioral(df)
    df = compute_continuity_score(df)
    df = compute_botnet_flag(df)

    rating = pd.to_numeric(df.get('rating', np.nan), errors='coerce').clip(0, 5).fillna(0)
    conf = pd.to_numeric(df.get('confidence', np.nan), errors='coerce').clip(0, 100).fillna(0)
    df['Xcrit'] = (((rating/5*2 - 2) + (conf/100*2 - 2)) / 2).astype(float)

    def msg_xcrit(r):
        msgs = []
        if r.get('rating', 0) >= RATING_HIGH:
            msgs.append("high analyst rating")
        if r.get('confidence', 0) >= CONFIDENCE_HIGH:
            msgs.append("high confidence")
        if not msgs:
            msgs.append("neutral analyst signals")
        return "; ".join(msgs)
    df['msg_xcrit'] = df.apply(msg_xcrit, axis=1)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# Modeling
# ─────────────────────────────────────────────────────────────────────────────
def train_unsupervised(df_feats, contamination=None, random_state=None):
    if contamination is None:
        contamination = MODEL_CONTAMINATION
    if random_state is None:
        random_state = MODEL_RANDOM_STATE

    X = df_feats.reindex(columns=FEATURE_COLS).fillna(0).values
    iso = IsolationForest(contamination=contamination, random_state=random_state)
    iso.fit(X)
    raw = -iso.decision_function(X)
    scaler = MinMaxScaler()
    df_feats = df_feats.copy()
    df_feats['anomaly_score'] = scaler.fit_transform(raw.reshape(-1,1)).ravel()

    q_anom = df_feats['anomaly_score'].quantile([0.25, 0.75]).to_dict()
    def msg_model(r):
        if r['anomaly_score'] >= q_anom[0.75]:
            return "model flags this as highly anomalous vs peers"
        if r['anomaly_score'] <= q_anom[0.25]:
            return "model sees this as typical vs peers"
        return "model sees moderate anomaly"
    df_feats['msg_model'] = df_feats.apply(msg_model, axis=1)

    return iso, df_feats

# ─────────────────────────────────────────────────────────────────────────────
# Severity Mapping (0–1000) with VT enhancement + hostname penalty
# ─────────────────────────────────────────────────────────────────────────────
def map_severity(
    df,
    score_col='anomaly_score',
    vt_field='vt_malicious_count',
    vt_weight=None,
    host_penalty_weight=None
):
    if vt_weight is None:
        vt_weight = VT_WEIGHT
    if host_penalty_weight is None:
        host_penalty_weight = HOST_PENALTY_WEIGHT

    df = df.copy()

    # VT normalization
    if vt_field in df.columns:
        vt_raw = pd.to_numeric(df[vt_field], errors='coerce').fillna(0).clip(lower=0)
    else:
        vt_raw = pd.Series(0, index=df.index, dtype=float)
    max_vt = float(vt_raw.max()) if len(vt_raw) else 0.0
    vt_norm = (vt_raw / max_vt) if max_vt > 0 else pd.Series(0.0, index=df.index, dtype=float)
    df['vt_norm'] = vt_norm

    # Base anomaly (0..1)
    base = pd.to_numeric(df.get(score_col, 0), errors='coerce').fillna(0).clip(0, 1)

    # clamp VT weight
    w = float(vt_weight)
    w = 0.0 if w < 0 else (1.0 if w > 1 else w)

    combined = ((1 - w) * base + w * vt_norm).astype(float)
    combined = combined.replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

    # Soft hostname penalty
    host_flag = pd.to_numeric(df.get('host_generic_4label_wordprefix', 0), errors='coerce').fillna(0).clip(0, 1)
    penalty_factor = (1.0 - host_penalty_weight * host_flag)
    combined = (combined * penalty_factor).clip(0, 1)

    df['combined_score'] = combined
    df['score_1000'] = np.rint(df['combined_score'] * 1000).astype(int).clip(0, 1000)

    df['severity'] = pd.cut(df['score_1000'], bins=BINS_1000, labels=BIN_LABELS, include_lowest=True, right=True)

    def msg_final(r):
        parts = [
            f"anomaly={r.get('anomaly_score', 0):.2f}",
            f"vt_norm={r.get('vt_norm', 0):.2f}",
            f"combined={r.get('combined_score', 0):.2f}",
            f"score={int(r.get('score_1000', 0))}",
            f"bin={r.get('severity')}"
        ]
        if r.get('vt_norm', 0) > 0:
            parts.append(f"(VT contributed with weight {w:.2f})")
        try:
            if int(r.get('host_generic_4label_wordprefix', 0)) == 1 and host_penalty_weight > 0:
                parts.append(f"(hostname pattern penalty −{int(host_penalty_weight*100)}%)")
        except Exception:
            pass

        # Reasoning summary
        reasons = []
        vt_n = float(r.get('vt_norm', 0))
        a = float(r.get('anomaly_score', 0))
        if vt_n == 0:
            reasons.append("no VT signal")
        elif vt_n > 0.5:
            reasons.append("strong VT signal")
        if a >= 0.75:
            reasons.append("very high anomaly score")
        elif a >= 0.5:
            reasons.append("high anomaly score")
        elif a <= 0.25:
            reasons.append("low anomaly score")
        if int(r.get('botnet_flag', 0)) > 0:
            reasons.append("botnet activity")
        fp = float(r.get('fp_rate', 0))
        if fp >= FP_RATE_HIGH_FLAG:
            reasons.append("high FP rate")
        elif fp <= FP_RATE_LOW_FLAG:
            reasons.append("low FP rate")

        sev = str(r.get('severity'))
        if sev in ['Low', 'Medium']:
            if "no VT signal" in reasons and "low anomaly score" in reasons:
                summary = f"{sev} severity — low VT signal and low anomaly score kept it down despite other positives."
            elif "no VT signal" in reasons and ("high anomaly score" in reasons or "very high anomaly score" in reasons):
                summary = f"{sev} severity — high anomaly score pushed it up despite no VT signal."
            elif "strong VT signal" in reasons and "low anomaly score" in reasons:
                summary = f"{sev} severity — strong VT signal kept it up despite low anomaly score."
            else:
                summary = f"{sev} severity — mix of factors."
        else:
            summary = f"{sev} severity — driven by {', '.join(reasons) if reasons else 'mixed factors'}."
        return " | ".join(parts) + " || " + summary

    df['msg_final'] = df.apply(msg_final, axis=1)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# Assemble explanations
# ─────────────────────────────────────────────────────────────────────────────
def assemble_explanations(df):
    df = df.copy()
    def join_msgs(row):
        msgs = [str(row[c]) for c in EXPLAIN_COLS if c in row and pd.notna(row[c]) and str(row[c]).strip()]
        return " ; ".join(msgs)
    df['explain'] = df.apply(join_msgs, axis=1)
    return df

# ─────────────────────────────────────────────────────────────────────────────
# Example Main (uses CONFIG values)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    df_raw = recent_tags  # assumed provided
    today = pd.Timestamp(datetime.utcnow().date())
    dates = [today - pd.Timedelta(days=i) for i in range(7)]

    daily = [{'indicator': ind, 'date': d, 'count': np.random.poisson(5)}
             for ind in df_raw['indicator'] for d in dates]
    daily_counts_df = pd.DataFrame(daily)

    owner_ids = df_raw.get('ownerId', pd.Series([], dtype=object))
    source_sightings_df = pd.DataFrame(
        [{'indicator': ind, 'ownerId': oid}
         for ind in df_raw['indicator'] for oid in pd.Series(owner_ids).dropna().unique()]
    )
    trust_map = {oid: int(np.random.choice([1,5,7,10])) for oid in pd.Series(owner_ids).dropna().unique()}

    df_feats = compute_features(df_raw, daily_counts_df, source_sightings_df, trust_map)
    model, df_scored = train_unsupervised(df_feats, contamination=MODEL_CONTAMINATION, random_state=MODEL_RANDOM_STATE)
    df_result = map_severity(df_scored, score_col='anomaly_score', vt_field='vt_malicious_count',
                             vt_weight=VT_WEIGHT, host_penalty_weight=HOST_PENALTY_WEIGHT)
    df_result = assemble_explanations(df_result)

    display(df_result[['indicator', 'score_1000', 'vt_malicious_count', 'severity', 'explain']])


,indicator,score_1000,vt_malicious_count,severity,explain
0,1.4.195.14,290,1,Medium,light recent sightings (low decayed volume); u...
1,101.89.174.236,264,6,Medium,upward trend in daily sightings ; broad corrob...
2,103.133.60.12,292,1,Medium,light recent sightings (low decayed volume); d...
3,103.149.86.208,242,5,Medium,upward trend in daily sightings ; broad corrob...
4,103.171.255.188,345,2,Medium,light recent sightings (low decayed volume); d...
...,...,...,...,...,...
835,vtext.com,460,0,Medium,light recent sightings (low decayed volume); d...
836,www.deepseek.com,267,0,Medium,light recent sightings (low decayed volume); u...
837,www.deepseek.com.cdn.cloudflare.net,265,0,Medium,light recent sightings (low decayed volume); d...
838,www.sthda.com,292,0,Medium,light recent sightings (low decayed volume); d...


In [98]:
import pandas as pd

DEFAULT_COLS = [
    "indicator", "type", "dateAdded", "lastObserved", "observations",
    "vt_malicious_count", "anomaly_score", "vt_norm", "combined_score",
    "score_1000", "severity"
]

def inspect_indicator(df: pd.DataFrame, indicator: str, cols=None, save_path: str = None):
    if cols is None:
        cols = [c for c in DEFAULT_COLS if c in df.columns]

    sub = df.loc[df["indicator"] == indicator].copy()
    if sub.empty:
        raise ValueError(f"Indicator not found in provided DataFrame: {indicator}")

    # Sort to get the "latest" entry if there are duplicates
    sort_keys = [c for c in ["lastObserved", "dateAdded"] if c in sub.columns]
    if sort_keys:
        sub = sub.sort_values(sort_keys, ascending=True)

    # Reorder columns (keep requested first, append any extras)
    existing_cols = [c for c in cols if c in sub.columns]
    other_cols = [c for c in sub.columns if c not in existing_cols]
    sub = sub[existing_cols + other_cols]

    if save_path:
        sub.to_csv(save_path, index=False)

    # Also return all rows for that indicator in case you want history
    return sub

# Example usage in notebook:
sub_df = inspect_indicator(df_result, indicator="1.4.195.14", save_path=None)
display(sub_df)


,indicator,type,dateAdded,lastObserved,observations,vt_malicious_count,anomaly_score,vt_norm,combined_score,score_1000,...,fp_rate,type_Address,type_EmailAddress,type_Host,type_Stripped URL,port_diversity,port_scan_confidence,continuity_score,botnet_flag,Xcrit
0,1.4.195.14,Address,2025-07-02,2025-08-07,2,1,0.445257,0.052632,0.248944,249,...,0.0,1,0,0,0,3,0,1,1,-0.51
